# Project: Decision tree from scratch 🌳
Write the CART algorithm yourself (Gini impurity, best split search, recursion), check it against scikit-learn on the breast-cancer dataset, then watch depth trade bias for variance.

Topic: [[Decision Trees]], [[Ensemble Methods]] · guide note: [[Project - Decision Tree from Scratch]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
data = load_breast_cancer(); X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)
print(X.shape, "classes:", data.target_names)

## 1. Gini impurity\n$G = 1 - \sum_k p_k^2$: 0 for a pure node, 0.5 for a 50/50 binary node. `y` holds integer labels.

In [ ]:
def gini(y):
    # TODO 1: 1 − Σ p_k²  (0 for an empty node)
    raise NotImplementedError  # TODO 1

## 2. Best split
Try every feature `j` and every threshold `t` among the feature's unique values (except the largest); the left child is `X[:, j] <= t`. Gain = parent Gini − weighted child Gini. Return `(feature, threshold, gain)` of the best split, or `(None, None, 0.0)` if nothing improves.

In [ ]:
def best_split(X, y):
    # TODO 2: exhaustive search over features and thresholds
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_tree():
    Xt = np.array([[1, 5], [2, 6], [3, 1], [4, 2.0]]); yt = np.array([0, 0, 1, 1])
    j, t, g = best_split(Xt, yt)
    return gini(np.array([0, 1])) == 0.5 and gini(np.array([1, 1])) == 0 and (j, t) in [(0, 2.0), (1, 2.0)] and abs(g - 0.5) < 1e-12 and best_split(Xt[:2], yt[:2])[0] is None
check("gini + split", _t_tree, "Toy data splits perfectly (gain 0.5); a pure node has nothing to split.")

def build(X, y, depth, max_depth, min_samples=2):
    j, t, gain = best_split(X, y) if depth < max_depth and len(y) >= min_samples else (None, None, 0)
    if j is None: return {"leaf": int(np.bincount(y).argmax())}
    left = X[:, j] <= t
    return {"j": j, "t": t, "L": build(X[left], y[left], depth + 1, max_depth), "R": build(X[~left], y[~left], depth + 1, max_depth)}
def predict_one(node, x):
    while "leaf" not in node: node = node["L"] if x[node["j"]] <= node["t"] else node["R"]
    return node["leaf"]
def predict(tree, X): return np.array([predict_one(tree, x) for x in X])

res = {}
if ready("gini + split"):
    for d in (1, 2, 3, 5, 8):
        tree = build(X_tr, y_tr, 0, d); sk = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
        res[d] = (np.mean(predict(tree, X_tr) == y_tr), np.mean(predict(tree, X_te) == y_te), sk.score(X_te, y_te))
        print(f"depth {d}: train {res[d][0]:.3f}  test {res[d][1]:.3f}   (sklearn test {res[d][2]:.3f})")
    root = build(X_tr, y_tr, 0, 1); print("root split:", data.feature_names[root["j"]], "<=", round(root["t"], 3))
    check("matches sklearn", lambda: abs(res[3][1] - res[3][2]) < 0.03 and res[8][0] == 1.0,
          "Depth-3 test accuracy should be within 3 points of sklearn, and depth 8 should fit the training set perfectly.")

<details><summary>▶ Solution</summary>

```python
def gini(y):
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    return float(1 - np.sum(p ** 2))
```

```python
def best_split(X, y):
    best = (None, None, 0.0)
    parent = gini(y)
    for j in range(X.shape[1]):
        for t in np.unique(X[:, j])[:-1]:
            left = X[:, j] <= t
            gain = parent - (left.mean() * gini(y[left]) + (~left).mean() * gini(y[~left]))
            if gain > best[2] + 1e-12:
                best = (j, float(t), float(gain))
    return best
```
</details>

## Stretch goals
- Use midpoints between sorted unique values as thresholds (what sklearn does) and see if you match it exactly.
- Bagging: average 50 of your trees on bootstrap samples (with random feature subsets = a random forest). How much does test accuracy improve?
- Replace Gini with entropy ([[Information Theory]]). Does the tree change?